In [1]:
"""
Phase 1 - Step 2 & 3: Train/Test Split + Baseline Random Forest
------------------------------------------------------------------
IMPORTANT CONCEPT: we split by BATTERY, not by random rows.
Why? If we split randomly, cycles from the SAME battery could end up
in both train and test - the model could "memorize" that battery's
curve instead of learning general degradation patterns. Splitting by
battery tests whether the model can predict SoH for a battery it has
NEVER seen before - a much more honest test.
"""
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import joblib

df = pd.read_csv("battery_dataset_features.csv")

# -------------------------------------------------------------
# Choose the input features (X) and the target to predict (y)
# These match your project brief: cycle count, avg discharge temp,
# depth of discharge, charge rate, time since last full charge
# + a couple of extra signals the data naturally gives us (temp, resistance)
# -------------------------------------------------------------
FEATURE_COLS = [
    "Discharge_Index",            # cycle count (age of battery in discharges)
    "Ambient_Temperature",        # room temp during test
    "Max_Temp_Reached",           # peak temp reached (extra useful signal)
    "Charge_Rate_Proxy",          # how fast it drained
    "Time_Since_Reset_Cycles",    # staleness of last impedance check
    "Internal_Resistance_Re",     # internal resistance (physics-linked to aging)
]
TARGET_COL = "SoH"

# -------------------------------------------------------------
# Split by Battery_ID, STRATIFIED by ambient temperature group.
# Why stratify: batteries were tested at very different temps
# (4C cold, 24C room, 43C hot). A random split can accidentally
# put ALL cold or ALL hot batteries into the test set, which makes
# the model look worse than it is (it never saw that temp regime
# during training at all). Stratifying keeps every temperature
# group represented on both sides of the split - a fairer test.
# -------------------------------------------------------------
battery_temp = df.groupby("Battery_ID")["Ambient_Temperature"].first()

test_batteries = []
train_batteries = []
np.random.seed(42)
for temp_group, batteries_in_group in battery_temp.groupby(battery_temp):
    ids = list(batteries_in_group.index)
    n_test = max(1, round(len(ids) * 0.2))  # ~20% of each temp group goes to test
    chosen_test = list(np.random.choice(ids, size=n_test, replace=False))
    test_batteries += chosen_test
    train_batteries += [b for b in ids if b not in chosen_test]

print(f"Train batteries ({len(train_batteries)}): {train_batteries}")
print(f"Test batteries  ({len(test_batteries)}): {test_batteries}")

train_df = df[df["Battery_ID"].isin(train_batteries)]
test_df = df[df["Battery_ID"].isin(test_batteries)]

X_train, y_train = train_df[FEATURE_COLS], train_df[TARGET_COL]
X_test, y_test = test_df[FEATURE_COLS], test_df[TARGET_COL]

print(f"\nTrain rows: {len(X_train)}, Test rows: {len(X_test)}")

# -------------------------------------------------------------
# Train a baseline Random Forest Regressor
# Random Forest = many decision trees voting together, good default
# choice because it needs little tuning and handles non-linear
# relationships (like temp/resistance interacting) out of the box.
# -------------------------------------------------------------
model = RandomForestRegressor(
    n_estimators=200,   # number of trees in the forest
    max_depth=10,       # limit tree depth to avoid overfitting on small data
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# -------------------------------------------------------------
# Predict on the held-out (unseen) test batteries
# -------------------------------------------------------------
y_pred = model.predict(X_test)

# -------------------------------------------------------------
# Validate: RMSE and MAE
# RMSE = root mean squared error - punishes big mistakes more
# MAE  = mean absolute error - average "how far off" in plain terms
# Both are in the same units as SoH (0-1 scale), so multiply by 100
# to read them as "percentage points of SoH error"
# -------------------------------------------------------------
rmse = np.sqrt(mean_squared_error(y_test, y_pred)) * 100
mae = mean_absolute_error(y_test, y_pred) * 100

print(f"\n--- Validation Results (on unseen batteries) ---")
print(f"RMSE: {rmse:.2f} percentage points of SoH")
print(f"MAE:  {mae:.2f} percentage points of SoH")

# -------------------------------------------------------------
# Feature importance - which inputs mattered most to the model?
# Useful for your report: shows the model isn't just guessing randomly.
# -------------------------------------------------------------
importances = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
print("\nFeature importances:")
print(importances)

# Save model and test predictions for the next step (plotting)
joblib.dump(model, "soh_random_forest_model.pkl")
test_df = test_df.copy()
test_df["Predicted_SoH"] = y_pred
test_df.to_csv("test_predictions.csv", index=False)

print("\nSaved model -> soh_random_forest_model.pkl")
print("Saved predictions -> test_predictions.csv")

Train batteries (25): ['B0045', 'B0046', 'B0047', 'B0048', 'B0051', 'B0053', 'B0054', 'B0055', 'B0056', 'B0042', 'B0043', 'B0006', 'B0007', 'B0018', 'B0025', 'B0026', 'B0027', 'B0028', 'B0033', 'B0036', 'B0039', 'B0040', 'B0030', 'B0031', 'B0032']
Test batteries  (7): [np.str_('B0049'), np.str_('B0041'), np.str_('B0044'), np.str_('B0034'), np.str_('B0038'), np.str_('B0005'), np.str_('B0029')]

Train rows: 2072, Test rows: 654

--- Validation Results (on unseen batteries) ---
RMSE: 12.00 percentage points of SoH
MAE:  6.96 percentage points of SoH

Feature importances:
Charge_Rate_Proxy          0.721285
Discharge_Index            0.118260
Internal_Resistance_Re     0.088193
Max_Temp_Reached           0.057700
Ambient_Temperature        0.011554
Time_Since_Reset_Cycles    0.003008
dtype: float64

Saved model -> soh_random_forest_model.pkl
Saved predictions -> test_predictions.csv
